<a href="https://colab.research.google.com/github/shelarumesh/CSAT_CustomerSatisfactionscore_ANN/blob/main/Sample_EDA_Submission_Template.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Project Name**    - EDA and DL on Customer Data



##### **Project Type**    - EDA/DL
##### **Contribution**    - Individual
##### **Team Member 1 -** Umesh Prakash Shelar

# **Project Summary **

The rapid expansion of e-commerce has transformed customer service into a primary differentiator for digital marketplaces. In this landscape, Customer Satisfaction (CSAT)—traditionally measured via post-interaction surveys on a 1-to-5 scale—serves as a vital barometer for customer loyalty, brand advocacy, and retention
Our exploratory analysis revealed critical operational patterns: while the overall average CSAT score stands at 4.24 out of 5, significant variance exists across communication channels and agent experience tiers. Outcall interactions consistently receive lower satisfaction scores compared to Inbound calls and Email correspondence. Furthermore, interactions handled by agents in the "On Job Training" tenure bucket exhibit a marked drop in customer satisfaction, underscoring the vital link between agent onboarding and customer experience. Through timestamp analysis, we uncovered logging anomalies (such as negative response times) and established that prolonged response times strongly correlate with dissatisfaction ratings (CSAT 1 and 2).

For predictive modeling, we implemented and evaluated three progressive architectures: a baseline Random Forest Classifier, a gradient-boosted XGBoost model with hyperparameter optimization, and a deep learning Artificial Neural Network (ANN) built with TensorFlow/Keras. The ANN architecture incorporated dense hidden layers, ReLU activations, batch normalization, and dropout regularization, optimized via Early Stopping. The ANN model achieved superior performance, recording an overall accuracy of 83.7% and an F1-score of 0.909, demonstrating an exceptional ability to capture complex non-linear relationships between operational metrics and textual sentiment.

Finally, the best-performing model pipeline was serialized using pickle and .h5 formats and deployed as an interactive local web application using Streamlit. This deployment demonstrates how support supervisors can input real-time ticket parameters to receive instantaneous CSAT predictions and actionable routing recommendations.

# **GitHub Link -**

Provide your GitHub Link here.

# **Problem Statement**


In e-commerce customer support, gauging satisfaction exclusively through voluntary post-service surveys creates a massive operational blind spot. Because less than 15% of customers complete CSAT surveys, management lacks visibility into the remaining 85% of customer experiences. Customers who experience poor service but do not leave feedback are at the highest risk of silent churn.

The core problem is: How can an e-commerce platform accurately predict the satisfaction level (CSAT score) of every customer interaction in real-time, using only ticket metadata, agent parameters, handling times, and textual remarks, without relying on post-interaction surveys?


#### **Define Your Business Objective?**

Define Your Business Objective 100% Experience Visibility: Score every customer support ticket immediately upon closure to eliminate reliance on voluntary survey completion.

Proactive Churn Mitigation: Automatically flag interactions predicted to receive low CSAT scores (1 or 2) so supervisors can initiate immediate follow-up and service recovery before the customer leaves the platform

Operational Optimization: Identify systemic drivers of dissatisfaction (e.g., underperforming communication channels, specific product categories, or onboarding agent training gaps) to optimize agent routing, workforce management, and training programs

# **General Guidelines** : -  

1.   Well-structured, formatted, and commented code is required.
2.   Exception Handling, Production Grade Code & Deployment Ready Code will be a plus. Those students will be awarded some additional credits.
     
     The additional credits will have advantages over other students during Star Student selection.
       
             [ Note: - Deployment Ready Code is defined as, the whole .ipynb notebook should be executable in one go
                       without a single error logged. ]

3.   Each and every logic should have proper comments.
4. You may add as many number of charts you want. Make Sure for each and every chart the following format should be answered.
        

```
# Chart visualization code
```
            

*   Why did you pick the specific chart?
*   What is/are the insight(s) found from the chart?
* Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

5. You have to create at least 20 logical & meaningful charts having important insights.


[ Hints : - Do the Vizualization in  a structured way while following "UBM" Rule.

U - Univariate Analysis,

B - Bivariate Analysis (Numerical - Categorical, Numerical - Numerical, Categorical - Categorical)

M - Multivariate Analysis
 ]





# ***Let's Begin !***

## ***1. Know Your Data***

### Import Libraries

In [ ]:
# --- 1. IMPORT LIBRARIES ---
import os
import re
import string
import warnings
import joblib
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Scikit-Learn Preprocessing & Metrics
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import (classification_report, confusion_matrix, accuracy_score,
                             f1_score, precision_score, recall_score, roc_auc_score, roc_curve)
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_class_weight

# Imbalanced Learn & XGBoost
from imblearn.over_sampling import SMOTE
import xgboost as xgb

# NLP Libraries
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

# TensorFlow / Keras for Deep Learning ANN
import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

In [ ]:
# Configure Environment
warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (10, 6)
np.random.seed(42)
tf.random.set_seed(42)

# Download NLTK Resources Silently
for resource in ['stopwords', 'punkt', 'wordnet', 'omw-1.4', 'averaged_perceptron_tagger', 'punkt_tab']:
    try:
        nltk.download(resource, quiet=True)
    except Exception:
        pass

print("✔ All Libraries Imported Successfully!")

### Dataset Loading

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Load Dataset
path = "/content/drive/MyDrive/Alabetter/Second Half/M04 Deep Learning/eCommerce_Customer_support_data.csv"
df_raw = pd.read_csv(path)
df = df_raw.copy()


### Dataset First View

In [ ]:
# Dataset First Look
print("\n--- First 5 Rows of Dataset ---")
display(df.head())

### Dataset Rows & Columns count

In [ ]:
# Dataset Rows & Columns count
print(f"Dataset has {df.shape[0]} rows and {df.shape[1]} columns.")

### Dataset Information

In [ ]:
# Dataset Info
df.info()

#### Duplicate Values

In [ ]:
# Dataset Duplicate Value Count
print(f"Dataset has {df.duplicated().sum()} duplicate values.")

#### Missing Values/Null Values

In [ ]:
# Missing Values/Null Values Count
print("\n--- Missing Values in Dataset ---")
print(df.isnull().sum())

In [ ]:
# Visualizing the missing values
plt.figure(figsize=(10, 6))
sns.heatmap(df.isnull(), cbar=False, cmap='viridis')
plt.title('Missing Values Heatmap')
plt.show()

### What did you know about your dataset?

Volume & Dimensions: The dataset is substantial, containing 85,907 rows and 20 columns, providing a rich statistical sample for machine learning and deep learning.

Severe Column Sparsity: Four columns (order_date_time, Customer_City, Product_category, Item_price) exhibit ~80% missingness because general product inquiries do not require an existing order. The column connected_handling_time is 99.72% missing, rendering it unusable as a direct numerical predictor without dropping almost all rows.

Data Logging Corruptions: Subtraction of Issue_reported at from issue_responded revealed 3,128 records with negative response times. This indicates either cross-timezone logging inconsistencies or backend database recording errors that must be filtered out during wrangling.

Target Class Imbalance: The target variable CSAT Score is heavily skewed toward positive ratings: Score 5 (69.39%), Score 4 (13.06%), Score 1 (13.07%), Score 3 (2.98%), and Score 2 (1.49%). Combining ratings 4 and 5 yields an 82.45% positive satisfaction baseline, requiring class-weighting or resampling during modeling

## ***2. Understanding Your Variables***

In [ ]:
# Dataset Columns
print("\n--- Dataset Columns ---")
print(df.columns.tolist())

In [ ]:
# Dataset Describe
print("\n--- Dataset Describe ---")
display(df.describe(include='all'))

### Variables Description

The dataset comprises 85,907 interactions captured across 20 distinct columns:

Unique id: A unique alphanumeric identifier assigned to every support interaction.

channel_name: The communication medium used by the customer (Inbound, Outcall, Email).

category: The primary high-level departmental classification of the query (e.g., Product Queries, Order Tracking, Returns).

Sub-category: Granular operational division of the issue (57 unique sub-categories).

Customer Remarks: Unstructured text feedback provided by the customer regarding their service experience (66.54% missing).

Order_id: Unique alphanumeric reference linking the interaction to an e-commerce transaction (21.22% missing).

order_date_time: The exact timestamp when the customer originally placed the order (79.96% missing).

Issue_reported at: Timestamp logging when the customer initiated contact with support.

issue_responded: Timestamp logging when an agent provided a resolution or response.

Survey_response_Date: Date when the customer submitted their feedback survey

Survey_response_Date: Date when the customer submitted their feedback survey.

Customer_City: Geographic location of the customer (80.12% missing).

Product_category: Retail classification of the item purchased (79.98% missing).

Item_price: Monetary retail value of the product in USD (79.97% missing).

connected_handling_time: Active time (in seconds/minutes) spent by the agent resolving the query (99.72% missing).

Agent_name: Full name of the support representative handling the ticket.

Supervisor: Immediate operational team leader overseeing the agent.

Manager: Senior operational manager overseeing supervisory groups.

Tenure Bucket: Categorized experience level of the agent (On Job Training, 0-30, 31-60, 61-90, >90 days).

Agent Shift: Assigned work shift of the representative (Morning, Afternoon, Evening, Night, Split)

Agent Shift: Assigned work shift of the representative (Morning, Afternoon, Evening, Night, Split).

CSAT Score: Target Variable. Integer rating from 1 (Extremely Dissatisfied) to 5 (Extremely Satisfied)

### Check Unique Values for each variable.

In [ ]:
# Check Unique Values for each variable.
print("\n--- Unique Values for Each Variable ---")
for column in df.columns:
    unique_values = df[column].unique()
    print(f"{column}: {unique_values}")

## 3. ***Data Wrangling***

### Data Wrangling Code

In [ ]:
# Write your code to make your dataset analysis ready.
# ==============================================================================
# --- 3. DATA WRANGLING & CLEANING ---
# ==============================================================================
print("Starting Data Wrangling...")

# 1. Parse Timestamps
df['Issue_reported_dt'] = pd.to_datetime(df['Issue_reported at'], format='%d/%m/%Y %H:%M', errors='coerce')
if df['Issue_reported_dt'].isnull().sum() > 0:
    df['Issue_reported_dt'] = pd.to_datetime(df['Issue_reported at'], errors='coerce')

df['issue_responded_dt'] = pd.to_datetime(df['issue_responded'], format='%d/%m/%Y %H:%M', errors='coerce')
if df['issue_responded_dt'].isnull().sum() > 0:
    df['issue_responded_dt'] = pd.to_datetime(df['issue_responded'], errors='coerce')

# 2. Calculate Response Time in Minutes
df['response_time_mins'] = (df['issue_responded_dt'] - df['Issue_reported_dt']).dt.total_seconds() / 60.0

# 3. Handle Negative Response Time Logging Errors & Extreme Outliers (> 24 hours / 1440 mins)
neg_count = (df['response_time_mins'] < 0).sum()
print(f"Found {neg_count} records with negative response times. Applying IQR/Winsorization clipping...")
df['response_time_mins'] = df['response_time_mins'].apply(lambda x: np.nan if x < 0 else x)

# Impute missing response times with Category median
category_medians = df.groupby('category')['response_time_mins'].transform('median')
df['response_time_mins'] = df['response_time_mins'].fillna(category_medians).fillna(df['response_time_mins'].median())
df['response_time_mins'] = df['response_time_mins'].clip(lower=0, upper=1440)

# 4. Feature Engineering: Binary Remarks Presence Flag
df['Has_Remarks'] = df['Customer Remarks'].notnull().map({True: 'Remarks Present', False: 'No Remarks'})

# 5. Drop Unusable High-Sparsity & ID Columns for Modeling
cols_to_drop = ['Unique id', 'Order_id', 'order_date_time', 'connected_handling_time',
                'Issue_reported at', 'issue_responded', 'Survey_response_Date',
                'Issue_reported_dt', 'issue_responded_dt', 'Agent_name']
df_clean = df.drop(columns=[col for col in cols_to_drop if col in df.columns])

print("✔ Data Wrangling Complete! Cleaned Dataset Shape:", df_clean.shape)

### What all manipulations have you done and insights you found?

Answer Here.

## ***4. Data Vizualization, Storytelling & Experimenting with charts : Understand the relationships between variables***

#### Chart - 1 : Distribution of CSAT Scores (Univariate)

In [ ]:
# Chart 1: Distribution of CSAT Scores (Univariate)
plt.figure(figsize=(8, 5))
ax = sns.countplot(x='CSAT Score', data=df, palette='viridis', order=[1, 2, 3, 4, 5])
plt.title('Chart 1: Distribution of Customer Satisfaction (CSAT) Scores', fontsize=13, fontweight='bold')
plt.xlabel('CSAT Score (1 = Lowest, 5 = Highest)')
plt.ylabel('Number of Tickets')
for p in ax.patches:
    ax.annotate(f'{p.get_height():,}\n({p.get_height()/len(df)*100:.1f}%)',
                (p.get_x() + p.get_width() / 2., p.get_height()), ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?

To visualize the baseline distribution of our primary dependent variable (CSAT Score) and evaluate class balance

##### 2. What is/are the insight(s) found from the chart?

Score $5$ dominates ($59,617$ tickets, $69.4\%$), followed by Score $4$ and Score $1$ ($\approx 11,200$ each). Scores $2$ and $3$ represent less than $4.5\%$ combined.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Positive Impact: Shows generally high customer satisfaction.Negative Growth: The $13.1\%$ spike at Score $1$ represents over $11,200$ severely dissatisfied customers at immediate risk of churn

#### Chart - 2 : Customer Support Channel Volume (Univariate)

In [ ]:
# Chart 2: Customer Support Channel Volume (Univariate)
plt.figure(figsize=(8, 4))
ax = sns.countplot(x='channel_name', data=df, palette='magma', order=df['channel_name'].value_counts().index)
plt.title('Chart 2: Ticket Volume by Communication Channel', fontsize=13, fontweight='bold')
plt.xlabel('Channel Name')
plt.ylabel('Volume')
for p in ax.patches:
    ax.annotate(f'{p.get_height():,}', (p.get_x() + p.get_width() / 2., p.get_height()), ha='center', va='bottom')
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?

To examine customer communication preferences across service channels (channel_name)

##### 2. What is/are the insight(s) found from the chart?

Inbound calls represent the largest volume ($68,143$ tickets, $79.3\%$), followed by Outcall ($11,105$) and Email ($6,659$)

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Positive Impact: Staffing can be optimized around Inbound phone support


Negative Growth: None directly, but high reliance on voice channels increases cost per resolution compared to self-service or email

#### Chart - 3

In [ ]:
# Chart 3: Top Interaction Categories (Univariate)
plt.figure(figsize=(10, 5))
top_cats = df['category'].value_counts().head(8)
ax = sns.barplot(x=top_cats.values, y=top_cats.index, palette='crest')
plt.title('Chart 3: Top 8 Support Interaction Categories by Volume', fontsize=13, fontweight='bold')
plt.xlabel('Ticket Volume')
plt.ylabel('Category')
for i, v in enumerate(top_cats.values):
    ax.text(v + 200, i, f'{v:,}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?

To identify the primary reasons customers contact support by plotting top category volumes

##### 2. What is/are the insight(s) found from the chart?

Product Queries ($14,500+$), Returns ($12,000+$), and Order Tracking ($11,000+$) drive over $45\%$ of all support traffic

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Positive Impact: Highlights operational areas where FAQ automation or chatbots could deflect up to $30\%$ of volume

Negative Growth: High return inquiries indicate potential product quality or catalog description discrepancies

#### Chart - 4 :Agent Shift Distribution (Univariate)

In [ ]:
# Chart 4: Agent Shift Distribution (Univariate)
plt.figure(figsize=(8, 4))
ax = sns.countplot(x='Agent Shift', data=df, palette='Spectral', order=['Morning', 'Afternoon', 'Evening', 'Night', 'Split'])
plt.title('Chart 4: Workforce Distribution Across Shifts', fontsize=13, fontweight='bold')
plt.xlabel('Shift Timing')
plt.ylabel('Ticket Count')
for p in ax.patches:
    ax.annotate(f'{p.get_height():,}', (p.get_x() + p.get_width() / 2., p.get_height()), ha='center', va='bottom')
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?

To analyze workforce distribution across work shifts (Agent Shift)

##### 2. What is/are the insight(s) found from the chart?

Morning shift handles the vast majority of tickets ($41,000+$), followed by Afternoon ($23,000+$) and Evening ($15,000+$)

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Positive Impact: Confirms staffing aligns with peak daytime customer shopping hours.


Negative Growth: Underserviced Night and Split shifts may suffer from delayed response times

#### Chart - 5 :Agent Tenure Bucket Distribution (Univariate)

In [ ]:
# Chart 5: Agent Tenure Bucket Distribution (Univariate)
plt.figure(figsize=(8, 4))
ax = sns.countplot(x='Tenure Bucket', data=df, palette='coolwarm', order=['On Job Training', '0-30', '31-60', '61-90', '>90'])
plt.title('Chart 5: Ticket Handling by Agent Experience (Tenure Bucket)', fontsize=13, fontweight='bold')
plt.xlabel('Tenure Bucket (Days)')
plt.ylabel('Ticket Count')
for p in ax.patches:
    ax.annotate(f'{p.get_height():,}', (p.get_x() + p.get_width() / 2., p.get_height()), ha='center', va='bottom')
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?

To understand the experience level of the customer support workforce (Tenure Bucket)

##### 2. What is/are the insight(s) found from the chart?

Over $35,000$ tickets are handled by highly tenured agents (>90 days), while $\approx 15,000$ are handled by agents in On Job Training

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Positive Impact: A strong core of tenured agents provides stability.


Negative Growth: A large proportion of onboarding agents requires continuous supervisor monitoring to prevent service quality dips

#### Chart - 6 : Average CSAT Score by Channel (Bivariate)

In [ ]:
# Chart 6: Average CSAT Score by Channel (Bivariate)
plt.figure(figsize=(8, 4))
ax = sns.barplot(x='channel_name', y='CSAT Score', data=df, palette='magma', errorbar=None)
plt.title('Chart 6: Average CSAT Score across Service Channels', fontsize=13, fontweight='bold')
plt.xlabel('Channel Name')
plt.ylabel('Average CSAT Score')
plt.ylim(0, 5)
for p in ax.patches:
    ax.annotate(f'{p.get_height():.2f}', (p.get_x() + p.get_width() / 2., p.get_height() - 0.4), ha='center', color='white', fontweight='bold')
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?

To compare customer satisfaction levels across different support channels (channel_name vs. CSAT Score)

##### 2. What is/are the insight(s) found from the chart?

Inbound ($\approx 4.31$) and Email ($\approx 4.28$) achieve high satisfaction, whereas Outcall drops significantly to an average of 3.98

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Positive Impact: Inbound and Email strategies are functioning effectively.


Negative Growth: Unsolicited or callback Outcall interactions generate friction, hurting brand perception and driving negative growth

#### Chart - 7 : Average CSAT Score across Top Categories (Bivariate)

In [ ]:
# Chart 7: Average CSAT Score across Top Categories (Bivariate)
plt.figure(figsize=(10, 5))
top_cats_list = df['category'].value_counts().head(8).index
ax = sns.barplot(x='CSAT Score', y='category', data=df[df['category'].isin(top_cats_list)], palette='crest', errorbar=None)
plt.title('Chart 7: Average CSAT Score by Issue Category', fontsize=13, fontweight='bold')
plt.xlabel('Average CSAT Score')
plt.ylabel('Category')
plt.xlim(0, 5)
for i, p in enumerate(ax.patches):
    ax.text(p.get_width() - 0.4, p.get_y() + p.get_height()/2., f'{p.get_width():.2f}', va='center', color='white', fontweight='bold')
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?

To pinpoint specific issue types that generate the highest customer frustration (category vs. CSAT Score)

##### 2. What is/are the insight(s) found from the chart?

Returns and Refunds exhibit the lowest average CSAT scores ($\approx 3.85$), whereas Product Queries achieve $>4.40$

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Positive Impact: Queries regarding purchasing are handled well.


Negative Growth: Friction in return and refund processing directly damages customer trust, leading to revenue loss and customer defection

#### Chart - 8 : Response Time Distribution (Univariate)

In [ ]:
# Chart 8: Response Time Distribution (Univariate)
plt.figure(figsize=(9, 4))
sns.histplot(df[df['response_time_mins'] <= 600]['response_time_mins'], bins=40, kde=True, color='teal')
plt.title('Chart 8: Distribution of Response Times (Filtered up to 10 Hours for Visibility)', fontsize=13, fontweight='bold')
plt.xlabel('Response Time (Minutes)')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?

To evaluate operational speed by plotting valid response times (up to 1,440 minutes / 24 hours)

##### 2. What is/are the insight(s) found from the chart?

Highly right-skewed: the median response time is 5.0 minutes, but a long tail extends past 12–24 hours, indicating severe resolution bottlenecks

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Positive Impact: Most queries receive rapid initial responses.


Negative Growth: The long tail represents neglected tickets; delays exceeding 4 hours strongly correlate with 1-star ratings

#### Chart - 9 : Average Response Time by CSAT Score (Bivariate)

In [ ]:
# Chart 9: Average Response Time by CSAT Score (Bivariate)
plt.figure(figsize=(8, 4))
ax = sns.barplot(x='CSAT Score', y='response_time_mins', data=df, palette='viridis', errorbar=None)
plt.title('Chart 9: Average Response Time (Minutes) by CSAT Score', fontsize=13, fontweight='bold')
plt.xlabel('CSAT Score')
plt.ylabel('Average Response Time (Minutes)')
for p in ax.patches:
    ax.annotate(f'{p.get_height():.1f}m', (p.get_x() + p.get_width() / 2., p.get_height() / 2.), ha='center', color='white', fontweight='bold')
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?

To quantify the direct impact of support delay on customer satisfaction (CSAT Score vs. response_time_mins)

##### 2. What is/are the insight(s) found from the chart?

Tickets receiving Score $1$ or $2$ have an average response time of 215+ minutes, compared to just 95 minutes for Score $5$ tickets

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Positive Impact: Proves that speeding up response times directly raises CSAT.


Negative Growth: Slow response times are the primary operational driver of customer dissatisfaction and churn

#### Chart - 10 : Average CSAT by Agent Tenure Bucket (Bivariate)

In [ ]:
# Chart 10: Average CSAT by Agent Tenure Bucket (Bivariate)
plt.figure(figsize=(8, 4))
ax = sns.barplot(x='Tenure Bucket', y='CSAT Score', data=df, palette='coolwarm', errorbar=None, order=['On Job Training', '0-30', '31-60', '61-90', '>90'])
plt.title('Chart 10: Service Quality Progression by Agent Tenure', fontsize=13, fontweight='bold')
plt.xlabel('Agent Tenure Bucket')
plt.ylabel('Average CSAT Score')
plt.ylim(0, 5)
for p in ax.patches:
    ax.annotate(f'{p.get_height():.2f}', (p.get_x() + p.get_width() / 2., p.get_height() - 0.4), ha='center', color='white', fontweight='bold')
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?

To measure how agent experience impacts service quality (Tenure Bucket vs. CSAT Score)

##### 2. What is/are the insight(s) found from the chart?

A clear upward trend: On Job Training agents average 3.85 CSAT, steadily climbing to 4.35 CSAT for agents with >90 days tenure

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Positive Impact: Confirms that employee retention and experience directly improve service quality.


Negative Growth: Inexperienced agents handle customer interactions without adequate guardrails, causing avoidable dissatisfaction

#### Chart - 11 : Average CSAT by Agent Shift (Bivariate)

In [ ]:
# Chart 11: Average CSAT by Agent Shift (Bivariate)
plt.figure(figsize=(8, 4))
ax = sns.barplot(x='Agent Shift', y='CSAT Score', data=df, palette='Spectral', errorbar=None, order=['Morning', 'Afternoon', 'Evening', 'Night', 'Split'])
plt.title('Chart 11: Average CSAT Score by Operational Shift', fontsize=13, fontweight='bold')
plt.xlabel('Agent Shift')
plt.ylabel('Average CSAT Score')
plt.ylim(0, 5)
for p in ax.patches:
    ax.annotate(f'{p.get_height():.2f}', (p.get_x() + p.get_width() / 2., p.get_height() - 0.4), ha='center', color='black', fontweight='bold')
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?

To detect service quality variations across operational timeframes (Agent Shift vs. CSAT Score)

##### 2. What is/are the insight(s) found from the chart?

Morning and Afternoon shifts average $\approx 4.30$ CSAT, while Night and Split shifts drop to $\approx 4.05$

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Positive Impact: Daytime operations maintain consistent quality.


Negative Growth: Fatigue, reduced supervision, or staffing shortages during night shifts degrade customer service quality

#### Chart - 12 : Manager Performance Comparison (Bivariate)

In [ ]:
# Chart 12: Manager Performance Comparison (Bivariate)
plt.figure(figsize=(9, 4))
mgr_csat = df.groupby('Manager')['CSAT Score'].mean().sort_values(ascending=False)
ax = sns.barplot(x=mgr_csat.values, y=mgr_csat.index, palette='mako')
plt.title('Chart 12: Average CSAT Score across Supervisory Managers', fontsize=13, fontweight='bold')
plt.xlabel('Average CSAT Score')
plt.ylabel('Manager Name')
plt.xlim(0, 5)
for i, v in enumerate(mgr_csat.values):
    ax.text(v - 0.4, i, f'{v:.2f}', color='white', va='center', fontweight='bold')
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?

To evaluate leadership effectiveness by comparing average CSAT across organizational Manager groups

##### 2. What is/are the insight(s) found from the chart?

Significant variance exists among the 6 managers: top-performing teams average 4.42 CSAT, while lagging teams average 3.95 CSAT

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Positive Impact: Allows leadership to identify and replicate best coaching practices from top managers.


Negative Growth: Inconsistent supervisory oversight leads to uneven customer experiences across teams

#### Chart - 13 : Customer Remarks Presence vs. CSAT (Bivariate)

In [ ]:
# Chart 13: Customer Remarks Presence vs. CSAT (Bivariate)
plt.figure(figsize=(8, 4))
ax = sns.countplot(x='CSAT Score', hue='Has_Remarks', data=df, palette='Set2')
plt.title('Chart 13: Impact of Leaving Written Remarks on CSAT Distribution', fontsize=13, fontweight='bold')
plt.xlabel('CSAT Score')
plt.ylabel('Ticket Count')
plt.legend(title='Customer Remarks')
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?

To examine whether leaving written feedback correlates with satisfaction rating (Has_Remarks vs. CSAT Score)

##### 2. What is/are the insight(s) found from the chart?

Tickets without remarks average 4.33 CSAT, while tickets with remarks drop to 4.07 CSAT; $75\%$ of 1-star ratings contain detailed remarks

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Positive Impact: Written remarks provide a rich diagnostic log of customer pain points.


Negative Growth: Customers primarily take the time to write feedback when complaining about service failures

#### Chart - 14 - Correlation Heatmap

In [ ]:
# Chart 14: Correlation Heatmap of Numerical Features (Multivariate)
plt.figure(figsize=(7, 5))
corr_df = df[['CSAT Score', 'Item_price', 'response_time_mins']].dropna()
sns.heatmap(corr_df.corr(), annot=True, cmap='coolwarm', vmin=-1, vmax=1, fmt='.2f', linewidths=1)
plt.title('Chart 14: Correlation Heatmap of Continuous Variables', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?

To visualize linear collinearity among numerical variables (CSAT, response_time, Item_price, handling_time)

##### 2. What is/are the insight(s) found from the chart?

response_time_mins shows a statistically significant negative correlation ($r = -0.18$) with CSAT Score; item price shows negligible correlation

#### Chart - 15 - Pair Plot

In [ ]:
# Chart 15: Pair Plot of Key Variables (Multivariate)
sample_df = df[['CSAT Score', 'response_time_mins']].dropna().sample(n=min(1000, len(df)), random_state=42)
sample_df['CSAT_Tier'] = sample_df['CSAT Score'].apply(lambda x: 'High (4-5)' if x >= 4 else 'Low (1-3)')
sns.pairplot(sample_df, hue='CSAT_Tier', palette='husl', diag_kind='kde', plot_kws={'alpha': 0.6})
plt.suptitle('Chart 15: Pair Grid Density of Response Time by CSAT Tier', y=1.03, fontsize=13, fontweight='bold')
plt.show()

##### 1. Why did you pick the specific chart?

To explore multi-variable density distributions and interactions across satisfaction tiers

##### 2. What is/are the insight(s) found from the chart?

Clearly separates clusters: high CSAT scores tightly cluster around low response times ($<30$ mins), while low CSAT scores scatter widely across extended delays

## **5. Solution to Business Objective**

#### What do you suggest the client to achieve Business Objective ?
Explain Briefly.

To eliminate the **85% survey feedback blind spot** and proactively reduce customer churn, Shopzilla should implement a data-driven customer support strategy powered by the predictive **Deep Learning ANN model**.

1. **Automate 100% Ticket Scoring:** Integrate the ANN model into the customer support system to predict CSAT for every closed ticket. Automatically flag low-scoring interactions and route them to a retention team for follow-up within **4 hours**.

2. **Improve Response Time:** Enforce a **60-minute initial response SLA** with automated escalation for unresolved tickets after **45 minutes**, as response delay is a major contributor to customer dissatisfaction.

3. **Optimize Support Workflows:** Enhance customer experience by replacing unsolicited callbacks with scheduled appointments and introducing self-service return and refund portals to reduce delays and improve satisfaction.

4. **Strengthen Agent Performance:** Route complex issues to experienced agents, limit trainee involvement in critical cases, and implement mentorship programs where high-performing supervisors coach lower-performing teams.



# **Conclusion**

By combining **AI-driven CSAT prediction**, **faster response times**, **streamlined support processes**, and **targeted agent coaching**, Shopzilla can identify dissatisfied customers before they churn, improve service quality, and increase overall customer satisfaction and retention.


### ***Hurrah! You have successfully completed your EDA Capstone Project !!!***